# 07 — Certified sampler inside the real Hugging Face pipeline: commit, stop and self-conditioning

Notebooks 02–04 only replaced the **commit** decision. The stock code still did two more full-vocabulary passes every step:

| Where (real source) | What it does every step | Full-vocab? |
|---|---|---|
| `EntropyBoundScheduler.step` (diffusers) | divide by T, softmax, sample, entropy → commit rule | yes |
| `DiffusionGemmaPipeline.__call__` (diffusers), stop check | `Categorical(pred_logits).entropy().mean() < 0.005` and argmax unchanged | yes (recomputes the same entropies) |
| `DiffusionGemmaDecoderModel.forward` (transformers), self-conditioning | `softmax(pred_logits) @ embed_tokens.weight * sqrt(2816)` | yes: a [256, 262144] × [262144, 2816] matmul |

This notebook patches all three, **without editing any library file**:

* `CertifiedEBScheduler`: certified commit from top-k entropy ranges [lo, hi], refining only ambiguous slots; samples from top-k with an exact tail branch.
* A pipeline subclass whose `__call__` is *your installed* stock `__call__` with four small text edits (each must match exactly once or it stops with an error). It prints the diff so you can see every change.
* A wrapper on `decoder.self_conditioning.forward` that injects our self-conditioning vector: `full` | `zero` | `topk` (tail `drop` / `renorm` / `mean`).

**Order of runs**

1. Load (same as notebook 02).
2. Test A: patched pipeline with the stock scheduler gives **identical tokens** to the original pipeline. Also, `sc='full'` (our injected self-conditioning) gives identical tokens. This shows the patch itself changes nothing.
3. Test B (shadow): the stock sampler drives generation; on every step we also compute the certified commit/stop and the top-k self-conditioning and count disagreements. Expected: 0 commit mismatches and 0 stop mismatches, plus the refinement % (the cost) and the self-conditioning error per tail option.
4. GSM8K: stock vs certified (self-conditioning full / top-k drop / renorm / mean) vs z=0 vs naive top-k renorm. Accuracy, steps per block, peak memory.

Timings here are **not meaningful** (the model is split over GPUs). Use this notebook for correctness and steps; speed goes in notebook 05.

In [1]:
import os, sys, subprocess
if 'torch' in sys.modules:
    raise RuntimeError('Restart the kernel, then run this cell before importing PyTorch.')

N_GPUS = int(os.environ.get('NB_N_GPUS', 4))   # split the ~52 GB of bf16 weights over this many GPUs
NEED_GB = 64
MANUAL_GPUS = None    # e.g. '0,1,2,3'; None = auto-pick by free memory (use another set for a parallel job)

if MANUAL_GPUS is None:
    rows = subprocess.run(['nvidia-smi', '--query-gpu=index,memory.free', '--format=csv,noheader,nounits'],
                          capture_output=True, text=True, check=True).stdout.strip().splitlines()
    if os.environ.get('SLURM_JOB_ID') and os.environ.get('CUDA_VISIBLE_DEVICES'):   # under Slurm: only the GPUs this job was given
        rows = [r for r in rows if r.split(', ')[0] in os.environ['CUDA_VISIBLE_DEVICES'].split(',')]
    free = sorted(((int(i), int(f) / 1024) for i, f in (r.split(', ') for r in rows)), key=lambda x: -x[1])
    chosen = [i for i, _ in free[:N_GPUS]]
    total = sum(f for _, f in free[:N_GPUS])
    print('free GB per GPU:', {i: round(f, 1) for i, f in free})
    if total < NEED_GB:
        print(f'WARNING: only {total:.0f} GB free on the {N_GPUS} chosen GPUs; loading will likely fail.')
    MANUAL_GPUS = ','.join(map(str, sorted(chosen)))
os.environ['CUDA_VISIBLE_DEVICES'] = MANUAL_GPUS
# Test A compares runs token by token; without this, cuBLAS and some kernels are not run-to-run deterministic
DETERMINISTIC = True
if DETERMINISTIC:
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print('Using physical GPUs', MANUAL_GPUS)

free GB per GPU: {6: 30.4, 1: 28.1, 5: 28.1, 7: 28.1, 2: 24.0, 3: 21.8, 0: 16.7, 4: 13.2}
Using physical GPUs 1,5,6,7


In [5]:
import time, re, json, inspect, difflib
import torch, transformers, diffusers
if DETERMINISTIC:
    torch.use_deterministic_algorithms(True, warn_only=True)   # warn instead of failing on ops without a deterministic version
    torch.backends.cudnn.benchmark = False
import pandas as pd
print('torch', torch.__version__, '| transformers', transformers.__version__, '| diffusers', diffusers.__version__)

# dg_certified.py must sit next to this notebook
import importlib, dg_certified as dc
importlib.reload(dc)

from transformers import DiffusionGemmaForBlockDiffusion, AutoProcessor
from diffusers import DiffusionGemmaPipeline
EntropyBoundScheduler, CertifiedEBScheduler = dc.make_scheduler_class()
CertifiedPipeline = dc.make_patched_pipeline_class(save_source_to='patched_call_source.py')

# show exactly what was changed in the installed stock __call__
orig = inspect.getsource(DiffusionGemmaPipeline.__call__)
new = open('patched_call_source.py').read()
import textwrap
for l in difflib.unified_diff(textwrap.dedent(orig).split('\n'), new.split('\n'), lineterm='', n=0):
    if l.startswith(('+', '-')) and not l.startswith(('+++', '---')):
        print(l.rstrip())

torch 2.14.1+cu130 | transformers 5.20.2 | diffusers 0.41.0
-@torch.no_grad()
-@replace_example_docstring(EXAMPLE_DOC_STRING)
+        self._rt.reset_block()
-            self_conditioning_logits = pred_logits
+            self_conditioning_logits = self._rt.prepare_self_conditioning(scheduler_output, pred_logits)
-                next_argmax_canvas = pred_logits.argmax(dim=-1)
+                next_argmax_canvas = self._rt.stop_argmax(scheduler_output, pred_logits)
-                confident = torch.distributions.Categorical(logits=pred_logits.float()).entropy().mean(-1) < (
-                    confidence_threshold
-                )
+                confident = self._rt.stop_confident(scheduler_output, pred_logits, confidence_threshold, stable)


The diff above should show only the decorator lines plus the four hooks: `reset_block`, `prepare_self_conditioning`, `stop_argmax` and `stop_confident`. If you get a 'matched 0 times' error, your diffusers version differs. Send me the output of `print(inspect.getsource(DiffusionGemmaPipeline.__call__))`.

In [6]:
MODEL_ID = 'google/diffusiongemma-26B-A4B-it'
DTYPE = torch.bfloat16
HEADROOM_GB = 5      # logits are [B, 256, 262144] fp32 = 268 MB per sequence, and the stock code makes several copies
max_memory = {d: f'{max(int(torch.cuda.mem_get_info(d)[0] / 2**30) - HEADROOM_GB, 0)}GiB'
              for d in range(torch.cuda.device_count())}
print('max_memory:', max_memory)
t0 = time.time()
model = DiffusionGemmaForBlockDiffusion.from_pretrained(MODEL_ID, dtype=DTYPE, device_map='auto',
                                                        max_memory=max_memory, low_cpu_mem_usage=True)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f'loaded in {time.time() - t0:.0f}s')
offloaded = {k: v for k, v in (getattr(model, 'hf_device_map', None) or {}).items() if v in ('cpu', 'disk')}
assert not offloaded, f'modules offloaded to CPU/disk: {list(offloaded)[:8]}'

N_STEPS = 48
SCHED_KW = dict(entropy_bound=0.1, t_max=0.8, t_min=0.4, num_inference_steps=N_STEPS)   # same as notebook 02
decoder = model.model.decoder
print('embed_scale =', float(decoder.embed_tokens.embed_scale), '| embedding table', tuple(decoder.embed_tokens.weight.shape),
      decoder.embed_tokens.weight.dtype, decoder.embed_tokens.weight.device)

pipe_orig = DiffusionGemmaPipeline(model=model, scheduler=EntropyBoundScheduler(**SCHED_KW), processor=processor)
pipe = CertifiedPipeline(model=model, scheduler=EntropyBoundScheduler(**SCHED_KW), processor=processor)

max_memory: {0: '16GiB', 1: '17GiB', 2: '7GiB', 3: '17GiB'}


Loading weights: 100%|██████████| 1047/1047 [00:38<00:00, 27.17it/s] 


loaded in 49s
embed_scale = 53.0 | embedding table (262144, 2816) torch.bfloat16 cuda:0


## Configurations

Each config = scheduler mode + self-conditioning mode + stop mode.

* `sched`: `stock` (the parent step, unchanged) | `certified` | `naive` (renormalized top-k entropy: the ablation, not exact)
* `sc`: `stock` (pass `pred_logits` through, as the original) | `full` (we compute the same softmax @ E and inject it) | `zero` (z = 0 baseline) | `topk`
* `stop`: `stock` | `certified` | `naive`

`certified` + `sc='full'` should match stock in decisions and accuracy (sampling uses different random numbers, so texts differ; that is expected).

In [7]:
CONFIGS = {
    'stock (patched)':        dict(sched='stock',     sc='stock', stop='stock'),
    'stock, sc injected full':dict(sched='stock',     sc='full',  stop='stock'),
    'z=0 (no self-cond)':     dict(sched='stock',     sc='zero',  stop='stock'),
    'certified, sc full':     dict(sched='certified', sc='full',  stop='certified'),
    'certified, sc topk drop':   dict(sched='certified', sc='topk', sc_tail='drop',   stop='certified'),
    'certified, sc topk renorm': dict(sched='certified', sc='topk', sc_tail='renorm', stop='certified'),
    'certified, sc topk mean':   dict(sched='certified', sc='topk', sc_tail='mean',   stop='certified'),
    'naive topk renorm':      dict(sched='naive',     sc='topk', sc_tail='renorm', stop='naive'),
}

REFERENCE = 'stock'   # 'stock' = match the float32 stock decisions; 'exact' = match float64 entropies

def use(cfg, k=64, shadow=False, timing=False, reference=None):
    # Fresh scheduler + runtime for one configuration; returns the runtime (holds the counters).
    rt = dc.Runtime(decoder, sc=cfg['sc'], sc_tail=cfg.get('sc_tail', 'drop'), stop=cfg['stop'],
                    k=k, shadow=shadow, timing=timing, reference=reference or REFERENCE)
    sch = CertifiedEBScheduler(**SCHED_KW).configure(rt, mode=cfg['sched'], k=k)
    pipe.scheduler = sch
    pipe.attach(rt)
    return rt

def run(p, prompt, seed=0, gen_length=256):
    torch.cuda.synchronize(); t0 = time.time()
    out = p(prompt=prompt, gen_length=gen_length, num_inference_steps=N_STEPS,
            generator=torch.Generator('cuda').manual_seed(seed))
    torch.cuda.synchronize()
    return out, time.time() - t0

PROMPTS = {
    'math': 'A train travels 120 km in 1.5 hours, then 80 km in 1 hour. What is its average speed? Show your steps.',
    'code': 'Write a Python function that returns the k most frequent words in a string, with a docstring.',
    'chat': 'Give three practical tips for sleeping better, one sentence each.',
}

## Pad check on saved logits (run before Test B)

Uses the logits notebook 04 saved. For each reference ('stock' = float32 stock entropies, 'exact' = float64) and pad: `violations` must be 0; `refined_pct` is the cost. Pick the smallest pad with 0 violations and set it in `dc.PADS`.

In [ ]:
import glob
files = sorted(glob.glob('./runs/nb04/step_*.pt'))      # adjust if notebook 04 saved them elsewhere
Ls = [torch.load(f).float().cuda() for f in files]
taus = [0.8 - 0.4 * int(f[-5:-3]) / 48 for f in files]   # stock temperature schedule, N = 48
print(len(Ls), 'saved steps')
print(dc.pad_sweep(Ls, taus).to_string(index=False))

## Test A: the patch itself changes nothing

Same seed, same prompt:
* original pipeline vs patched pipeline (stock scheduler, `sc='stock'`) → **identical tokens** (tests the four edits)
* original pipeline vs `sc='full'` (our injected self-conditioning) → **identical tokens** (tests the injection hook)

If the second one differs only on some prompts, it is GPU non-determinism; rerun the original twice to check that it matches itself.

In [8]:
rows = []
for name, prompt in PROMPTS.items():
    pipe_orig.scheduler = EntropyBoundScheduler(**SCHED_KW)
    ref, _ = run(pipe_orig, prompt, seed=1)
    ref2, _ = run(pipe_orig, prompt, seed=1)
    use(CONFIGS['stock (patched)'])
    a, _ = run(pipe, prompt, seed=1)
    use(CONFIGS['stock, sc injected full'])
    b, _ = run(pipe, prompt, seed=1)
    rows.append(dict(prompt=name,
                     original_matches_itself=torch.equal(ref.sequences, ref2.sequences),
                     patched_identical=torch.equal(ref.sequences, a.sequences),
                     injected_full_identical=torch.equal(ref.sequences, b.sequences)))
print(pd.DataFrame(rows).to_string(index=False))
print('\nexample output:', ref.texts[0][:300])

 15%|█▍        | 7/48 [00:03<00:19,  2.12it/s]
/data/users/shreenis/private/VLA_experiments/dg_certified.py:261: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  mx = max(mx, float(blk.norm(dim=-1).max()))
 19%|█▉        | 9/48 [00:03<00:16,  2.34it/s]

prompt  original_matches_itself  patched_identical  injected_full_identical
  math                     True               True                     True
  code                    False              False                    False
  chat                    False              False                    False

example output: thought
To solve this problem, the most efficient approach is to use the `collections.Counter` class, which is designed specifically for counting hashable objects.

### The Solution

```python
from collections import Counter
import re

def get_k_most_frequent_words(text, k=5):
    """
    Identifies


## Test B: shadow run (the key correctness table)

The **stock** sampler drives generation. On every step we additionally compute, and compare:

| column | meaning | want |
|---|---|---|
| `sh_commit_mismatch_rows` | steps where the certified commit set ≠ stock commit set | 0 |
| `sh_stop_mismatch` | certified "mean entropy < 0.005" ≠ stock's | 0 |
| `sh_argmax_mismatch` | top-1 of top-k ≠ stock argmax (float ties only) | 0 or tiny |
| `sh_refined_commit_pct` | % of slots computed exactly for the commit (the cost) | low |
| `sh_refined_stop_pct` | % of slots computed exactly for the stop check | low |
| `sh_tail_mass_mean` | average probability outside the top-k | — |
| `sh_sc_relerr_*` | relative error of top-k self-conditioning vs full, per tail option | small; compare drop / renorm / mean |
| `sh_sc_bound_violations` | slots where the `drop` error exceeded m × max‖E‖ × √d | 0 |

Note: here the stop check is shadowed for every example on every step (the real certified path only checks examples whose argmax is stable, which is cheaper).

In [9]:
KS = [16, 64, 256, 1024]
shadow_rows = []
for k in KS:
    for name, prompt in PROMPTS.items():
        rt = use(CONFIGS['stock (patched)'], k=k, shadow=True)
        out, dt = run(pipe, prompt, seed=2)
        shadow_rows.append(dict(k=k, prompt=name, **rt.summary()))
shadow = pd.DataFrame(shadow_rows)
os.makedirs('./runs/nb07', exist_ok=True)
shadow.to_csv('./runs/nb07/shadow.csv', index=False)
cols = ['k', 'prompt', 'steps', 'sh_commit_mismatch_rows', 'sh_commit_mismatch_ties', 'sh_commit_mismatch_exact', 'sh_stop_mismatch_exact', 'sh_stop_mismatch', 'sh_argmax_mismatch',
        'sh_refined_commit_pct', 'sh_refined_stop_pct', 'sh_tail_mass_mean',
        'sh_sc_relerr_drop', 'sh_sc_relerr_renorm', 'sh_sc_relerr_mean', 'sh_sc_bound_violations']
pd.set_option('display.float_format', '{:.4f}'.format)
print(shadow[cols].to_string(index=False))

 12%|█▎        | 6/48 [00:02<00:19,  2.11it/s]

   k prompt  steps  sh_commit_mismatch_rows  sh_stop_mismatch  sh_argmax_mismatch  sh_refined_commit_pct  sh_refined_stop_pct  sh_tail_mass_mean  sh_sc_relerr_drop  sh_sc_relerr_renorm  sh_sc_relerr_mean  sh_sc_bound_violations
  16   math      8                        0                 0                   6                 0.1465               0.0000             0.0203             0.0195               0.0144             0.0184                     924
  16   code     17                        1                 0                  24                 0.6434               0.0000             0.0225             0.0209               0.0169             0.0199                    1891
  16   chat      6                        0                 0                   1                 1.0417               0.0000             0.0117             0.0101               0.0084             0.0099                     386
  64   math      8                        0                 0                   0       

**How to read it**

* Mismatches must be 0 at every k. If not, send me the row. (A rare mismatch at a float tie is possible: two slots whose entropies are equal to ~1e-7.)
* Refinement % is the headline cost number. It should fall as k grows. If it's still above a few % at k = 64, the pad may be too wide, or the early high-temperature steps may dominate. The next cell splits it by step.
* For self-conditioning, pick the tail option with the smallest error at k = 64. That's the one to carry into GSM8K.

In [10]:
# refinement by step index, k=64, math prompt: is the cost concentrated in the first (hot, T=0.8) steps?
rt = use(CONFIGS['stock (patched)'], k=64, shadow=True)
per_step = []
orig_commit = rt.shadow_commit
def logged(logits, T, b, keep):
    before = rt.c['sh_refined_commit']
    orig_commit(logits, T, b, keep)
    per_step.append(dict(step=len(per_step), T=round(T, 3),
                         refined_pct=100 * (rt.c['sh_refined_commit'] - before) / (logits.shape[0] * logits.shape[1]),
                         committed=int(keep.sum()), tail_mass=float(rt.last['st']['m'].mean())))
rt.shadow_commit = logged
out, _ = run(pipe, PROMPTS['math'], seed=2)
print(pd.DataFrame(per_step).to_string(index=False))

  0%|          | 0/48 [00:00<?, ?it/s]

 17%|█▋        | 8/48 [00:03<00:16,  2.45it/s]

 step      T  refined_pct  committed  tail_mass
    0 0.8000       0.0000         17     0.0076
    1 0.7920       0.0000         49     0.0001
    2 0.7830       0.0000         63     0.0001
    3 0.7750       0.0000        136     0.0000
    4 0.7670       0.0000        166     0.0000
    5 0.7580       1.1719        182     0.0000
    6 0.7500       0.0000        255     0.0000
    7 0.7420       0.0000        256     0.0000


## GSM8K: stock vs certified vs z = 0 vs naive

Same prompt format and seed-per-question as notebook 02. Results append to a CSV, so the run resumes if interrupted.
Start with `N_GSM = 50` to check everything end to end, then 250, then all 1,319.

What to expect:
* `certified, sc full` ≈ `stock` in accuracy and steps per block (same decisions; different random draws).
* `certified, sc topk *`: same commit/stop rule, but an approximate self-conditioning input, so accuracy and steps may move slightly. That's what this table measures.
* `z=0` and `naive topk renorm` are the lower baselines: naive should take fewer steps (it rushes) and lose accuracy.

In [11]:
from datasets import load_dataset
N_GSM = int(os.environ.get('NB_N_GSM', 50))
K = 64
RUN_CONFIGS = ['stock (patched)', 'certified, sc full', 'certified, sc topk drop', 'certified, sc topk renorm',
               'certified, sc topk mean', 'z=0 (no self-cond)', 'naive topk renorm']
if os.environ.get('NB_RUN_CONFIGS'):          # e.g. 'stock (patched);certified, sc full'
    RUN_CONFIGS = [c.strip() for c in os.environ['NB_RUN_CONFIGS'].split(';')]
RESULTS_CSV = f'./runs/nb07/gsm8k_{N_GSM}_k{K}.csv'
gsm = load_dataset('openai/gsm8k', 'main', split=f'test[:{N_GSM}]')

done = set()
if os.path.exists(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(prev.config, prev.q))
    print(f'resuming: {len(done)} generations already in {RESULTS_CSV}')

for name in RUN_CONFIGS:
    t_start, n_new = time.time(), 0
    for j, ex in enumerate(gsm):
        if (name, j) in done:
            continue
        rt = use(CONFIGS[name], k=K)
        prompt = ex['question'] + '\nSolve step by step, then give the final answer as a number.'
        n_dev = torch.cuda.device_count()
        base = [torch.cuda.memory_allocated(d) for d in range(n_dev)]
        for d in range(n_dev):
            torch.cuda.reset_peak_memory_stats(d)
        out, dt = run(pipe, prompt, seed=j, gen_length=512)
        pred, gold = dc.last_number(out.texts[0]), dc.gsm8k_gold(ex['answer'])
        n_tok = len(processor.tokenizer(out.texts[0])['input_ids'])
        s = rt.summary()
        row = dict(config=name, q=j, pred=pred, gold=gold, correct=dc.same_number(pred, gold),
                   steps=s['steps'], blocks=s['blocks'], tokens=n_tok, seconds=dt,
                   # memory generation adds on top of the loaded model, on the GPU where it is largest (the LM-head GPU)
                   peak_extra_gb=max(torch.cuda.max_memory_allocated(d) - base[d] for d in range(n_dev)) / 2**30,
                   refined_commit_pct=s.get('refined_commit_pct', 0.0),
                   refined_stop_pct=s.get('refined_stop_pct', 0.0),
                   tail_sample_pct=s.get('tail_sample_pct', 0.0))
        pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=not os.path.exists(RESULTS_CSV), index=False)
        n_new += 1
        if n_new % 10 == 0:
            el = time.time() - t_start
            print(f'{name}: q {j + 1}/{N_GSM} | {el / n_new:.1f}s per problem')
print('done')

 10%|█         | 10/96 [00:04<00:38,  2.21it/s]


stock (patched): q 10/50 | 4.4s per problem


 12%|█▎        | 12/96 [00:04<00:31,  2.64it/s]


stock (patched): q 20/50 | 4.2s per problem


  9%|▉         | 9/96 [00:04<00:39,  2.19it/s]


stock (patched): q 30/50 | 4.8s per problem


... (60 outputs left out)

 16%|█▌        | 15/96 [00:05<00:31,  2.60it/s]


naive topk renorm: q 40/50 | 4.4s per problem


  9%|▉         | 9/96 [00:03<00:38,  2.25it/s]

naive topk renorm: q 50/50 | 4.4s per problem
done


In [12]:
res = pd.read_csv(RESULTS_CSV)
summary = res.groupby('config', sort=False).agg(
    n=('q', 'count'), accuracy=('correct', 'mean'), steps_per_block=('steps', 'sum'),
    tokens=('tokens', 'mean'), refined_commit_pct=('refined_commit_pct', 'mean'),
    refined_stop_pct=('refined_stop_pct', 'mean'), tail_sample_pct=('tail_sample_pct', 'mean'),
    **({'peak_extra_gb': ('peak_extra_gb', 'max')} if 'peak_extra_gb' in res else {}))
summary['steps_per_block'] = res.groupby('config', sort=False)['steps'].sum() / res.groupby('config', sort=False)['blocks'].sum()
summary['tokens_per_step'] = res.groupby('config', sort=False)['tokens'].sum() / res.groupby('config', sort=False)['steps'].sum()
summary = summary.reindex([c for c in RUN_CONFIGS if c in summary.index])
print(summary.to_string(float_format='{:.3f}'.format))

# paired comparison against stock on the same questions
ref = res[res.config == 'stock (patched)'].set_index('q')['correct']
for name in summary.index:
    if name == 'stock (patched)':
        continue
    other = res[res.config == name].set_index('q')['correct']
    both = ref.index.intersection(other.index)
    print(f'{name:28s} vs stock: both right {int((ref[both] & other[both]).sum()):4d} | only stock {int((ref[both] & ~other[both]).sum()):3d} '
          f'| only this {int((~ref[both] & other[both]).sum()):3d} | n={len(both)}')

                            n  accuracy  steps_per_block  tokens  refined_commit_pct  refined_stop_pct  tail_sample_pct  peak_gb  tokens_per_step
config                                                                                                                                           
stock (patched)            50     0.980            7.237 260.040               0.000             0.000            0.000   16.429           23.640
certified, sc full         50     1.000            7.400 257.820               0.394             0.000            0.181   16.430           23.227
certified, sc topk drop    50     0.940            7.039 258.720               0.376             0.000            0.205   16.430           24.179
certified, sc topk renorm  50     0.980            6.779 255.720               0.343             0.000            0.209   16.430           24.494
certified, sc topk mean    50     1.000            7.156 258.880               0.502             0.000            0.206   16

**What goes in the report from this notebook**

1. Test A: the patch is a no-op when switched off (identical tokens).
2. The shadow table: 0 commit/stop mismatches across k, the refinement % by k, and the self-conditioning error by tail option.
3. GSM8K: accuracy, steps per block and tokens per step for stock vs certified-full (should match), the top-k self-conditioning variants (what we pay for skipping the 262k matmul), and the z = 0 / naive baselines.

`tokens_per_step` here is per sequence: total text tokens ÷ total denoising steps. The paper's TPF also adds the K − 1 append passes. Compute both if you compare to its Table 4.